# 00 · Build `res_data/enet_all_results.h5` from the ElasticNet LOO run

`01. res_analysis_loo` and `02_enet_scT_comparison` need **one row per protein × LOO sample**
(`protein`, `LOO sample`, `true`, `predicted`, `params_dict`, …).

- `<P>_LOO.parquet` (written by `new_pipeline.py --full_output`, the default in `new_run_enet.sh`) carries those rows.
- `<P>_best.parquet` holds only the single best LOO split per protein, so it **cannot** rebuild per-sample predictions.
  It is merged here into `res_data/enet_best_summary.csv` (coefficients, hyper-parameters, `mse_all_others`).

Run once after copying the LOO results into `../new_pipeline_loo/data/results_<norm>/`.

In [ ]:
# >>> pRNA figure routing (added by the pRNA update; keep as the first code cell)
# Every figure is shown inline. Every file written with savefig (png, pdf, svg) is redirected to
# figures_pdf/<notebook name>/ next to this notebook; sub-folders of the original path are preserved.
import os as _fig_os
from pathlib import Path as _FigPath
import matplotlib as _fig_mpl
import matplotlib.pyplot as _fig_plt
from matplotlib.figure import Figure as _FigFigure

NOTEBOOK_NAME = "00_build_enet_all_results"
FIG_DIR = (_FigPath.cwd() / "figures_pdf" / NOTEBOOK_NAME).resolve()
FIG_DIR.mkdir(parents=True, exist_ok=True)


def _route_figure_path(fname):
    p = _FigPath(_fig_os.fspath(fname))
    full = p if p.is_absolute() else (_FigPath.cwd() / p)
    try:
        full.resolve().relative_to(FIG_DIR)
        return fname                                  # already inside this notebook's figure folder
    except ValueError:
        pass
    try:
        rel = _FigPath(_fig_os.path.relpath(full.resolve(), _FigPath.cwd()))
    except ValueError:                                # e.g. another drive on Windows
        rel = _FigPath(p.name)
    parts = [x for x in rel.parts if x not in ("..", ".", "figures_pdf", NOTEBOOK_NAME)]
    target = FIG_DIR.joinpath(*parts) if parts else FIG_DIR / p.name
    target.parent.mkdir(parents=True, exist_ok=True)
    return target


if not getattr(_FigFigure.savefig, "_prna_routed", False):
    _fig_orig_savefig = _FigFigure.savefig

    def _fig_routed_savefig(self, fname, *args, **kwargs):
        if isinstance(fname, (str, bytes, _FigPath)) or hasattr(fname, "__fspath__"):
            fname = _route_figure_path(fname)
        return _fig_orig_savefig(self, fname, *args, **kwargs)

    _fig_routed_savefig._prna_routed = True
    _FigFigure.savefig = _fig_routed_savefig

    _fig_orig_close = _fig_plt.close

    def _fig_close_and_show(fig=None):
        """Display figures that are closed before Jupyter could render them, then close."""
        import inspect as _fig_inspect
        _caller = _fig_inspect.currentframe().f_back
        _caller_mod = (_caller.f_globals.get("__name__", "") if _caller is not None else "")
        if _caller_mod.startswith(("matplotlib", "ipykernel")):   # backend housekeeping, not user code
            return _fig_orig_close(fig)
        if "inline" in str(_fig_mpl.get_backend()).lower():
            from IPython.display import display as _fig_display
            open_nums = set(_fig_plt.get_fignums())
            if fig is None:
                figs = [_fig_plt.gcf()] if open_nums else []
            elif isinstance(fig, str) and fig == "all":
                figs = [_fig_plt.figure(n) for n in sorted(open_nums)]
            elif isinstance(fig, _FigFigure):
                figs = [fig] if getattr(fig, "number", None) in open_nums else []
            elif isinstance(fig, int):
                figs = [_fig_plt.figure(fig)] if fig in open_nums else []
            else:
                figs = []
            for _f in figs:
                if not getattr(_f, "_prna_displayed", False):
                    _f._prna_displayed = True
                    _fig_display(_f)
        return _fig_orig_close(fig)

    _fig_plt.close = _fig_close_and_show

    _fig_orig_show = _fig_plt.show

    def _fig_show_and_mark(*args, **kwargs):
        for _n in _fig_plt.get_fignums():                    # shown now -> never re-displayed on close
            _fig_plt.figure(_n)._prna_displayed = True
        return _fig_orig_show(*args, **kwargs)

    _fig_plt.show = _fig_show_and_mark

print("Figures are saved to", FIG_DIR)

In [ ]:
from pathlib import Path
import json
import pandas as pd

NORMALIZATION = 'zscore'                                   # which LOO run to analyse: zscore | none | ranking | binning
LOO_DIR = Path('../new_pipeline_loo/data') / f'results_{NORMALIZATION}'
OUT_DIR = Path('res_data'); OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_H5 = OUT_DIR / 'enet_all_results.h5'
OUT_BEST = OUT_DIR / 'enet_best_summary.csv'
assert LOO_DIR.is_dir(), f'LOO result folder not found: {LOO_DIR.resolve()}'
print('Reading from', LOO_DIR.resolve())

In [ ]:
def read_any(path):
    return pd.read_parquet(path) if path.suffix == '.parquet' else pd.read_csv(path)

def files_for(pattern):
    # prefer parquet, fall back to the CSV written when no parquet engine was available
    found = {}
    for f in sorted(LOO_DIR.glob(pattern + '.csv')) + sorted(LOO_DIR.glob(pattern + '.parquet')):
        found[f.name.rsplit('.', 1)[0]] = f
    return list(found.values())

def to_json(v):
    if isinstance(v, str):
        return v
    if isinstance(v, dict):
        return json.dumps({k: x for k, x in v.items() if x is not None})
    return json.dumps(v) if v is not None else '{}'

loo_files, best_files = files_for('*_LOO'), files_for('*_best')
print(f'{len(loo_files)} *_LOO files | {len(best_files)} *_best files')

In [ ]:
if not loo_files:
    raise FileNotFoundError(
        'No *_LOO files found. Per-sample predictions only exist when new_pipeline.py runs with --full_output '
        '(FULL_OUTPUT=1 in new_run_enet.sh). *_best files alone cannot rebuild enet_all_results.h5.')

loo = pd.concat([read_any(f) for f in loo_files], ignore_index=True)
required = {'protein', 'LOO sample', 'true', 'predicted', 'params_dict'}
assert required <= set(loo.columns), f'missing columns: {required - set(loo.columns)}'
for col in ('params_dict', 'features_coef_dict'):
    if col in loo.columns:
        loo[col] = loo[col].map(to_json)
loo['true'] = pd.to_numeric(loo['true'], errors='coerce')
loo['predicted'] = pd.to_numeric(loo['predicted'], errors='coerce')
loo['protein'] = loo['protein'].astype(str)
loo['LOO sample'] = loo['LOO sample'].astype(str)
dups = loo.duplicated(['protein', 'LOO sample']).sum()
assert dups == 0, f'{dups} duplicated protein x sample rows – mixed runs in {LOO_DIR}?'

loo.to_hdf(OUT_H5, key='data', mode='w', format='fixed')
print(f'Wrote {OUT_H5}: {loo.shape[0]:,} rows, {loo.protein.nunique():,} proteins, '
      f'{loo["LOO sample"].nunique():,} samples')

In [ ]:
if best_files:
    best = pd.concat([read_any(f) for f in best_files], ignore_index=True)
    for col in ('params_dict', 'feat_coeffs_dict'):
        if col in best.columns:
            best[col] = best[col].map(to_json)
    for col in ('zscore_genes', 'zscore_mu', 'zscore_sigma'):
        if col in best.columns:
            best[col] = best[col].map(lambda v: json.dumps(list(v)) if v is not None and not isinstance(v, str) else v)
    best.to_csv(OUT_BEST, index=False)
    print(f'Wrote {OUT_BEST}: {len(best):,} proteins')

# sanity check: the analysis notebooks read the file exactly like this
with pd.HDFStore(OUT_H5, mode='r') as store:
    check = pd.concat([store[k] for k in store.keys()], ignore_index=True)
print('Re-read OK:', check.shape, list(check.columns))